# Gợi ý việc làm bằng GNN đồ thị dị thể + trọng số động — Notebook chạy toàn bộ trên Kaggle
**Chạy từ đầu đến cuối trên Kaggle** (tiền xử lý → dựng đồ thị → huấn luyện B0–B4 → đánh giá → độ phức tạp → cắt tỉa → dị thể vs đồng nhất → xuất bộ nhãn tay).

**Cài đặt Kaggle:** Settings → *Accelerator = GPU T4 x2 / P100*, *Internet = On*. Internet chỉ dùng để `pip install` thư viện và tải **trọng số mô hình embedding e5** (không tải dữ liệu); tắt Internet thì notebook tự dùng TF-IDF + SVD thay cho e5.
**Dữ liệu:** notebook **không tải dữ liệu từ mạng** (không HuggingFace VietJobs, không clone GitHub). Thêm dataset `vn-job-rec-graph` bằng *Add Input*; ô mục 2 tự tìm trong `/kaggle/input`.
Dataset gồm `jobs.csv` (38.101 tin), `users.csv` (2.098 ứng viên), `interactions.csv` (lượt ứng tuyển mô phỏng + chia train/val/test), `hard_negatives.csv`, `meta.json` — tạo bằng `Job_Recommended/ml/build_training_dataset.py` (nguồn gốc: dữ liệu backend). Hướng dẫn chi tiết: `kaggle_readme.md`.
Kết quả nằm ở `/kaggle/working/results/` (và `results_bundle.zip`).

| Mô hình | Đồ thị | Tích chập | Trọng số quan hệ | Lọc cứng + xếp hạng lại |
|---|---|---|---|---|
| B0_SAGE_2nut | 2 loại nút (User, Job) | SAGE | — | — |
| B1_HeteroSAGE | 7 loại nút (6 + Đãi ngộ) | SAGE | — | — |
| B2_HeteroGATv2 (+ B2_HGT) | 7 loại nút (6 + Đãi ngộ) | GATv2 (HGT) | — | — |
| B3s_TrongSoTinh (đối chứng) | 7 loại nút (6 + Đãi ngộ) | GATv2 | tĩnh (tham số học được, không phụ thuộc cặp) | — |
| B3_TrongSoDong | 7 loại nút (6 + Đãi ngộ) | GATv2 | động w = σ(g_r(p_u, φ_r(u,j))) | — |
| B4a / B4b | như B3 | | động | ✓ lọc cứng + xếp hạng lại (a: tuyến tính; b: LambdaRank/LightGBM) |

Mỗi mô hình được báo cáo ở 2 biến thể: `raw` (không lọc) và `+mask` (lọc cứng tỉnh/kinh nghiệm/bằng cấp khi xếp hạng) để tách riêng đóng góp của mô hình và của bộ lọc.


## 0. Cài thư viện

In [ ]:
import sys, subprocess, os
if os.path.exists('/kaggle'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric', 'sentence-transformers', 'lightgbm'], check=False)
import torch, torch_geometric; print('torch', torch.__version__, '| PyG', torch_geometric.__version__, '| GPU:', torch.cuda.is_available())

## 1. Cấu hình
`SMOKE=1` (biến môi trường) chạy thu nhỏ để kiểm tra luồng; Kaggle mặc định chạy đầy đủ (3 seed, tối đa 100 epoch, dừng sớm theo NDCG@20 trên validation).

In [ ]:
import os, sys, re, json, time, math, random, unicodedata, subprocess, warnings, gc, copy
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

IN_KAGGLE = os.path.exists("/kaggle")
SMOKE = os.environ.get("SMOKE", "0") == "1"      # SMOKE=1: chạy thu nhỏ để kiểm tra nhanh luồng chạy
WORK = Path("/kaggle/working") if IN_KAGGLE else Path(os.environ.get("WORK", "./work"))
CACHE, OUT = WORK / "cache", WORK / "results"
for p in (WORK, CACHE, OUT):
    p.mkdir(parents=True, exist_ok=True)

CFG = dict(
    seeds=[0] if SMOKE else [0, 1, 2],           # số lần chạy lặp (mean ± std)
    max_jobs=3000 if SMOKE else None,            # SMOKE: số job ngẫu nhiên thêm vào ngoài các job có nhãn của ứng viên được chọn
    max_users=300 if SMOKE else None,
    emb_model="intfloat/multilingual-e5-small",  # sentence encoder đa ngôn ngữ (hỗ trợ tiếng Việt)
    emb_max_len=256,
    exp_tol=1.0,        # dung sai kinh nghiệm (năm): job đòi <= kinh nghiệm ứng viên + dung sai
    sal_ratio=0.8,      # ràng buộc mềm: lương cao nhất của job >= sal_ratio * lương mong muốn
    n_hard_pool=100,    # số negative khó tối đa mỗi ứng viên (đọc từ hard_negatives.csv)
    d=64, layers=2, heads=2, dropout=0.1, lr=5e-3, wd=1e-5,
    epochs=int(os.environ.get("EPOCHS", 4 if SMOKE else 100)), batch=int(os.environ.get("BATCH", 1024)),   # 1024 ≈ 34 bước/epoch; 4096 quá ít bước cập nhật eval_every=2 if SMOKE else 5, patience=6,
    n_handlabel_users=20 if SMOKE else 100,
)
print("IN_KAGGLE =", IN_KAGGLE, "| SMOKE =", SMOKE, "| WORK =", WORK)
print(json.dumps(CFG, ensure_ascii=False))


## 2. Tải dữ liệu
Bộ dữ liệu đã chuẩn hóa sẵn (Kaggle dataset `vn-job-rec-graph`). Ô dưới tự tìm thư mục chứa `jobs.csv` + `interactions.csv` trong `/kaggle/input`; chạy ngoài Kaggle thì đặt biến môi trường `DATA_DIR`.


In [ ]:
# ---- Tìm thư mục dataset ----
def _find_data_dir():
    if os.environ.get("DATA_DIR"):
        return Path(os.environ["DATA_DIR"])
    roots = [Path("/kaggle/input"), Path("./kaggle_dataset"), Path("../kaggle_dataset"), Path(".")]
    for r in roots:
        if not r.exists():
            continue
        for f in sorted(r.rglob("interactions.csv")):
            if (f.parent / "jobs.csv").exists() and (f.parent / "meta.json").exists():
                return f.parent
    raise FileNotFoundError("Không tìm thấy dataset (jobs.csv, users.csv, interactions.csv, meta.json). "
                            "Kaggle: Add Input -> dataset vn-job-rec-graph; local: đặt DATA_DIR.")
DATA_DIR = _find_data_dir()
print("DATA_DIR =", DATA_DIR, {p.name: f"{p.stat().st_size/1e6:.1f} MB" for p in sorted(DATA_DIR.iterdir()) if p.is_file()})


## 3. Đọc dữ liệu đã chuẩn hóa
Toàn bộ khâu làm sạch đã làm ở `build_training_dataset.py`: khử trùng tin/ứng viên, khôi phục tỉnh từ địa chỉ (63 → 34 đơn vị theo NQ 202/2025/QH15), khôi phục ngành, nghề chung cho job và ứng viên (35 nghề), lương (triệu VND/tháng, *thỏa thuận* = thiếu), bằng cấp tối thiểu từ yêu cầu, cấp bậc, đãi ngộ 9 nhóm.


In [ ]:
# ===== Tham số dùng chung + đọc meta =====
def fold(s):
    """Bỏ dấu tiếng Việt, hạ chữ thường."""
    s = unicodedata.normalize("NFD", str(s).replace("đ", "d").replace("Đ", "D"))
    return "".join(c for c in s if unicodedata.category(c) != "Mn").lower().strip()

META = json.load(open(DATA_DIR / "meta.json", encoding="utf-8"))
UNIT_NAMES = META["unit_names"]; UNIT_IDX = {n: i for i, n in enumerate(UNIT_NAMES)}
UNITS = {n: tuple(ll) for n, ll in zip(UNIT_NAMES, META["unit_latlon"])}        # đơn vị -> (vĩ độ, kinh độ)
N_UNIT = len(UNIT_NAMES); LOC_ANY = N_UNIT; N_LOC = N_UNIT + 1                    # nút Địa bàn: 34 đơn vị + "Toàn quốc"
DEGREE_NAMES = {int(k): v for k, v in META["degree_levels"].items()}
BEN_NAMES = META["benefit_groups"]; N_BEN = len(BEN_NAMES)
_lat = np.radians([UNITS[n][0] for n in UNIT_NAMES]); _lon = np.radians([UNITS[n][1] for n in UNIT_NAMES])
_a = np.sin((_lat[:, None] - _lat[None]) / 2) ** 2 + np.cos(_lat[:, None]) * np.cos(_lat[None]) * np.sin((_lon[:, None] - _lon[None]) / 2) ** 2
DIST_KM = 2 * 6371 * np.arcsin(np.sqrt(_a))          # 34x34 (km, đường chim bay)

def parse_units(s):
    """'Hà Nội;TP. Hồ Chí Minh' -> [0, 26] ; 'Toàn quốc' -> [LOC_ANY] ; '' -> []"""
    if not isinstance(s, str) or not s.strip():
        return []
    l = [x.strip() for x in s.split(";") if x.strip()]
    return [LOC_ANY] if META["any_unit"] in l else sorted(UNIT_IDX[x] for x in l if x in UNIT_IDX)
print({k: META[k] for k in ("n_jobs", "n_users", "n_interactions", "split_counts")})
print("Độ phủ thuộc tính:", json.dumps(META["coverage"], ensure_ascii=False))


In [ ]:
# ===== Đọc bảng job / ứng viên / tương tác =====
raw_jobs = pd.read_csv(DATA_DIR / "jobs.csv", dtype={"job_id": str, "units": str, "ben_cats": str}, low_memory=False)
raw_users = pd.read_csv(DATA_DIR / "users.csv", dtype={"units": str}, low_memory=False)
raw_jobs["units"] = raw_jobs["units"].fillna(""); raw_users["units"] = raw_users["units"].fillna("")
INTER = pd.read_csv(DATA_DIR / "interactions.csv")
HARDN = pd.read_csv(DATA_DIR / "hard_negatives.csv")

if SMOKE:   # thu nhỏ: chọn ngẫu nhiên ứng viên, giữ job có nhãn của họ + một ít job ngẫu nhiên, đánh lại chỉ số
    _r = np.random.RandomState(0)
    _us = np.sort(_r.choice(len(raw_users), min(CFG["max_users"], len(raw_users)), replace=False))
    _js = set(INTER.loc[INTER.u_idx.isin(_us), "j_idx"]) | set(HARDN.loc[HARDN.u_idx.isin(_us), "j_idx"])
    _js = np.sort(list(_js | set(_r.choice(len(raw_jobs), min(CFG["max_jobs"], len(raw_jobs)), replace=False))))
    _um, _jm = {o: n for n, o in enumerate(_us)}, {o: n for n, o in enumerate(_js)}
    raw_users = raw_users.iloc[_us].reset_index(drop=True); raw_jobs = raw_jobs.iloc[_js].reset_index(drop=True)
    INTER = INTER[INTER.u_idx.isin(_um) & INTER.j_idx.isin(_jm)].assign(u_idx=lambda d: d.u_idx.map(_um), j_idx=lambda d: d.j_idx.map(_jm))
    HARDN = HARDN[HARDN.u_idx.isin(_um) & HARDN.j_idx.isin(_jm)].assign(u_idx=lambda d: d.u_idx.map(_um), j_idx=lambda d: d.j_idx.map(_jm))

# Bảng chuẩn dùng cho toàn notebook (lương quy về VND/tháng như các ô phía sau)
jobs = pd.DataFrame(dict(
    job_id=raw_jobs["job_id"], title=raw_jobs["title"], units=raw_jobs["units"].map(parse_units),
    exp_min=raw_jobs["exp_min"].astype(float), exp_max=raw_jobs["exp_max"].astype(float),
    sal_min=raw_jobs["sal_min"] * 1e6, sal_max=raw_jobs["sal_max"] * 1e6, sal_kind=raw_jobs["sal_kind"],
    deg_min=raw_jobs["deg_min"].astype(float), industry=raw_jobs["industry_group"], occupation=raw_jobs["occupation"],
    level=raw_jobs["level"].astype(int), ben_known=raw_jobs["ben_known"].astype(bool),
    ben_cats=raw_jobs["ben_cats"].fillna("").map(lambda s: [int(float(x)) for x in s.split(";") if x not in ("", "nan")]),
    text_full=raw_jobs["text_full"].fillna("").astype(str)))
users = pd.DataFrame(dict(
    user_id=raw_users["user_id"], age=raw_users["age"].astype(float), units=raw_users["units"].map(parse_units),
    exp_min=raw_users["exp_min"].astype(float), exp_max=raw_users["exp_max"].astype(float),
    sal_min=raw_users["sal_min"] * 1e6, sal_max=raw_users["sal_max"] * 1e6, sal_kind=raw_users["sal_kind"],
    deg_max=raw_users["deg_max"].astype(float), industry=raw_users["industry_group"], occupation=raw_users["occupation"],
    married=raw_users["married"].astype(bool), text_full=raw_users["text_full"].fillna("").astype(str)))
NU, NJ = len(users), len(jobs)
print(f"{NJ} job, {NU} ứng viên, {len(INTER)} lượt ứng tuyển, {len(HARDN)} negative khó")
for nm, df in (("job", jobs), ("user", users)):
    print(nm, "| có địa bàn:", f"{df['units'].map(bool).mean():.1%}", "| có KN:", f"{df['exp_min'].notna().mean():.1%}",
          "| có lương:", f"{(df['sal_min'].notna() | df['sal_max'].notna()).mean():.1%}",
          "| có bằng cấp:", f"{df['deg_min' if nm=='job' else 'deg_max'].notna().mean():.1%}")
print("Đãi ngộ: có mô tả", f"{jobs['ben_known'].mean():.1%}", "| số nhóm TB / job:", round(jobs["ben_cats"].map(len).mean(), 2))


## 4. Mảng ràng buộc (dùng chung cho nhãn, mask, đo vi phạm)
Vi phạm = có dữ liệu ở cả hai phía **và** không thỏa; thiếu dữ liệu không bị tính là vi phạm.

In [ ]:
import torch
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def _f(s): return torch.tensor(np.asarray(s, dtype=np.float32), device=DEV)
def _multihot(lists):
    m = np.zeros((len(lists), N_UNIT), np.float32)
    for i, l in enumerate(lists):
        for u in l:
            if u < N_UNIT: m[i, u] = 1
    return m
def _prim(lists):   # đơn vị chính: đầu danh sách; 34 = Toàn quốc; 35 = không rõ
    return np.array([(l[0] if l else 35) for l in lists], np.int64)

jobs["sal_hi"] = jobs["sal_max"].where(jobs["sal_max"].notna(), jobs["sal_min"])
T = dict(
    jx=_f(jobs["exp_min"]), jh=_f(jobs["sal_hi"] / 1e6), jd=_f(jobs["deg_min"]),
    jb=_f(jobs["ben_cats"].map(len) / N_BEN), jbk=_f(jobs["ben_known"].astype(float)),
    jneg=_f((jobs["sal_kind"] == "negotiable").astype(float)),
    jp=_f(_multihot(jobs["units"])), jany=_f(jobs["units"].map(lambda l: LOC_ANY in l)), jhas=_f(jobs["units"].map(bool)),
    jprim=torch.tensor(_prim(jobs["units"]), device=DEV),
    ux=_f(users["exp_max"]), us=_f(users["sal_min"] / 1e6), ud=_f(users["deg_max"]), uage=_f(users["age"]), umar=_f(users["married"].astype(float)),
    up=_f(_multihot(users["units"])), uany=_f(users["units"].map(lambda l: LOC_ANY in l)), uhas=_f(users["units"].map(bool)),
    uprim=torch.tensor(_prim(users["units"]), device=DEV),
)
DIST36 = np.zeros((36, 36), np.float32); DIST36[:34, :34] = DIST_KM; T["dist"] = torch.tensor(DIST36, device=DEV)
# ngưỡng hợp lý để tránh giá trị lương rác (VND/triệu)
T["jh"] = torch.where(T["jh"] > 2000, torch.full_like(T["jh"], float("nan")), T["jh"])
T["us"] = torch.where(T["us"] > 2000, torch.full_like(T["us"], float("nan")), T["us"])
T["jh"] = torch.where(T["jh"] <= 0, torch.full_like(T["jh"], float("nan")), T["jh"])
T["us"] = torch.where(T["us"] <= 0, torch.full_like(T["us"], float("nan")), T["us"])

def viol(uidx):
    """uidx: LongTensor[B] -> dict ràng buộc -> BoolTensor[B, NJ] (True = vi phạm; thiếu dữ liệu không tính vi phạm)."""
    u = uidx
    overlap = (T["up"][u] @ T["jp"].T) > 0
    p = (T["uhas"][u, None] > 0) & (T["uany"][u, None] == 0) & (T["jhas"][None] > 0) & (T["jany"][None] == 0) & ~overlap
    e = T["jx"][None] > (T["ux"][u, None] + CFG["exp_tol"])
    d = T["jd"][None] > T["ud"][u, None]
    s = T["jh"][None] < CFG["sal_ratio"] * T["us"][u, None]
    return dict(province=p, experience=e, degree=d, salary=s)   # so sánh với NaN -> False

def any_hard(v): return v["province"] | v["experience"] | v["degree"]
def any_all(v): return any_hard(v) | v["salary"]
print("Mảng ràng buộc sẵn sàng trên", DEV)


## 5. Embedding văn bản và nhãn
Nhãn lấy từ `interactions.csv`: **lượt ứng tuyển mô phỏng** theo mô hình hành vi (phù hợp nghề/ngành/kỹ năng + độ hấp dẫn của tin + sở thích tiềm ẩn chung theo nghề + phạt ràng buộc có cường độ phụ thuộc đặc điểm ứng viên; chọn bằng Gumbel-top-k), đã chia sẵn 70/15/15 mỗi ứng viên. Negative khó = job liên quan nhưng vi phạm ràng buộc cứng. Chi tiết: `meta.json → behaviour_model`.
**Lưu ý:** đây vẫn là dữ liệu tương tác mô phỏng (không có log thật) → xem thêm bộ nhãn tay ở mục 12.


In [ ]:
# ===== Embedding văn bản =====
def encode_texts(texts, tag):
    f = CACHE / f"emb_{tag}_{len(texts)}.npy"
    if f.exists():
        return np.load(f)
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer(CFG["emb_model"], device=str(DEV)); m.max_seq_length = CFG["emb_max_len"]
        if DEV.type == "cuda": m.half()
        E = m.encode(["passage: " + t for t in texts], batch_size=128, normalize_embeddings=True, show_progress_bar=True)
        print("Embedding:", CFG["emb_model"])
    except Exception as ex:
        print("!! Không dùng được sentence-transformers (", type(ex).__name__, ") -> dự phòng TF-IDF + SVD")
        return None
    np.save(f, E.astype(np.float32)); return E.astype(np.float32)

EU = encode_texts(users["text_full"].tolist(), "user"); EJ = encode_texts(jobs["text_full"].tolist(), "job")
if EU is None or EJ is None:   # dự phòng
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    tv = TfidfVectorizer(sublinear_tf=True, min_df=2, max_features=100000, ngram_range=(1, 2))
    X = tv.fit_transform(pd.concat([jobs["text_full"], users["text_full"]]).map(fold))
    Z = TruncatedSVD(128 if SMOKE else 256, random_state=0).fit_transform(X)
    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9
    EJ, EU = Z[:NJ].astype(np.float32), Z[NJ:].astype(np.float32)
print("EU", EU.shape, "EJ", EJ.shape)

# ===== Nhãn từ interactions.csv (đã chia train/val/test) =====
rng = np.random.RandomState(0)
split = {"train": {}, "val": {}, "test": {}}
for (u, sp), g in INTER.groupby(["u_idx", "split"]):
    split[sp][int(u)] = g.sort_values("pref_rank")["j_idx"].to_numpy(np.int64)
pos = {u: np.concatenate([split[s].get(u, np.array([], np.int64)) for s in split]) for u in range(NU)}
hard = {int(u): g["j_idx"].to_numpy(np.int64)[:CFG["n_hard_pool"]] for u, g in HARDN.groupby("u_idx")}
keep = [u for u in range(NU) if all(len(split[s].get(u, [])) > 0 for s in split)]
for s in split:
    split[s] = {u: split[s][u] for u in keep}
for u in range(NU):
    hard.setdefault(u, np.array([], np.int64))
print(f"Ứng viên có đủ train/val/test: {len(keep)}/{NU}; trung vị #dương = {np.median([len(pos[u]) for u in keep]):.0f}; "
      f"trung vị #negative khó = {np.median([len(hard[u]) for u in keep]):.0f}")
assert len(keep) >= 20, "Quá ít ứng viên có nhãn"
KEEP = np.array(keep)
tr_pairs = np.array([(u, j) for u in keep for j in split["train"][u]], dtype=np.int64)
print("Cặp train/val/test:", len(tr_pairs), sum(len(v) for v in split["val"].values()), sum(len(v) for v in split["test"].values()))
# kiểm tra nhanh: tỉ lệ vi phạm của nhãn dương so với job ngẫu nhiên
_ub = torch.tensor(KEEP[:512], device=DEV); _v = viol(_ub)
_pp = [(r, j) for r, u in enumerate(KEEP[:512]) for j in pos[u]]; _r_, _j_ = (torch.tensor(x, device=DEV) for x in zip(*_pp))
print("Vi phạm trong nhãn dương:", {c: round(float(_v[c][_r_, _j_].float().mean()), 4) for c in _v},
      "| job ngẫu nhiên:", {c: round(float(_v[c].float().mean()), 4) for c in _v})


## 6. Đồ thị 7 loại nút
User, Job, Kinh nghiệm, Địa bàn, Mức lương, Học vấn + Đãi ngộ (chỉ nối với job; cạnh `near` giữa các nút thuộc tính).

In [ ]:
# ===== Đồ thị 6 loại nút =====
ATTRS = ["exp", "loc", "sal", "deg", "ben"]     # "ben" (đãi ngộ) chỉ nối với job: ứng viên không có dữ liệu ưu tiên đãi ngộ
N_ATTR = dict(exp=6, loc=N_LOC, sal=8, deg=6, ben=N_BEN)     # exp: 5 khoảng + "không rõ"; sal: 7 khoảng + "thỏa thuận/không rõ"; deg: 5 mức + "không rõ"
def b_exp(v):  return 5 if pd.isna(v) else (0 if v <= 0 else 1 if v <= 2 else 2 if v <= 5 else 3 if v <= 10 else 4)
def b_sal(lo, hi):
    vals = [x for x in (lo, hi) if not pd.isna(x)]
    if not vals: return 7
    m = np.mean(vals) / 1e6
    return 7 if (m <= 0 or m > 2000) else int(np.searchsorted([5, 10, 15, 20, 30, 50], m, side="right"))
def b_deg(v):  return 5 if pd.isna(v) else int(v)

# cạnh chuẩn (job/user -> nút thuộc tính)
CAN = {}
for side, df, ex in (("user", users, "exp_max"), ("job", jobs, "exp_min")):
    dg = "deg_max" if side == "user" else "deg_min"
    sl = ("sal_min", "sal_min") if side == "user" else ("sal_min", "sal_max")
    ent = np.arange(len(df))
    CAN[(side, "exp")] = (ent, df[ex].map(b_exp).to_numpy())
    CAN[(side, "sal")] = (ent, np.array([b_sal(a, b) for a, b in zip(df[sl[0]], df[sl[1]])]))
    CAN[(side, "deg")] = (ent, df[dg].map(b_deg).to_numpy())
    s_, d_ = zip(*[(i, l) for i, ls in enumerate(df["units"]) for l in ls]) if len(df) else ([], [])
    CAN[(side, "loc")] = (np.array(s_, np.int64), np.array(d_, np.int64))
_bs, _bd = zip(*[(i, c) for i, cs in enumerate(jobs["ben_cats"]) for c in cs]) if jobs["ben_cats"].map(len).sum() else ([], [])
CAN[("job", "ben")] = (np.array(_bs, np.int64), np.array(_bd, np.int64))
# cạnh near giữa các nút thuộc tính: trọng số cạnh = độ gần
NEAR = {}
for a in ("exp", "sal", "deg"):
    n = N_ATTR[a] - (1 if a != "sal" else 1)     # bỏ nút "không rõ" ở cuối
    s = np.arange(n - 1); NEAR[a] = (np.r_[s, s + 1], np.r_[s + 1, s], np.ones(2 * (n - 1), np.float32))
k = 4; nn_idx = np.argsort(DIST_KM, 1)[:, 1:k + 1]
ls_ = np.repeat(np.arange(N_UNIT), k); ld_ = nn_idx.reshape(-1)
NEAR["loc"] = (ls_, ld_, np.exp(-DIST_KM[ls_, ld_] / 300).astype(np.float32))

TYPES_H = ["user", "job"] + ATTRS
def make_graph(kind, keep_w=None):
    """kind: 'b0' (2 loại nút) | 'hetero' (6 loại) | 'homo' (gộp 1 loại).
    keep_w: dict khóa cạnh chuẩn -> BoolArray giữ cạnh (dùng khi cắt tỉa)."""
    ei, ea = {}, {}
    def add(et, s, d, w=None):
        s, d = torch.as_tensor(s, device=DEV, dtype=torch.long), torch.as_tensor(d, device=DEV, dtype=torch.long)
        w = torch.ones(len(s), 1, device=DEV) if w is None else torch.as_tensor(w, device=DEV).float().view(-1, 1)
        ei[et] = torch.stack([s, d]); ea[et] = w
    # user-job (chỉ cạnh train)
    add(("user", "applies", "job"), tr_pairs[:, 0], tr_pairs[:, 1]); add(("job", "rev_applies", "user"), tr_pairs[:, 1], tr_pairs[:, 0])
    if kind != "b0":
        for (side, a), (s, d) in CAN.items():
            if keep_w is not None and (side, a) in keep_w:
                m = keep_w[(side, a)]; s, d = s[m], d[m]
            add((side, f"has_{a}", a), s, d); add((a, f"rev_has_{a}", side), d, s)
        for a, (s, d, w) in NEAR.items():
            if keep_w is not None and ("near", a) in keep_w:
                m = keep_w[("near", a)]; s, d, w = s[m], d[m], w[m]
            add((a, f"near_{a}", a), s, d, w)
    counts = dict(user=NU, job=NJ, **N_ATTR)
    types = ["user", "job"] if kind == "b0" else TYPES_H
    G = dict(kind=kind, types=types, n={t: counts[t] for t in types}, ei=ei, ea=ea)
    if kind == "homo":      # gộp: chỉ số nút = offset theo loại
        off, o = {}, 0
        for t in types: off[t] = o; o += counts[t]
        G["off"], G["N"] = off, o
        G["h_ei"] = torch.cat([torch.stack([e[0] + off[et[0]], e[1] + off[et[2]]]) for et, e in ei.items()], 1)
        G["h_ea"] = torch.cat(list(ea.values()), 0)
    return G
G_b0, G_het, G_homo = make_graph("b0"), make_graph("hetero"), make_graph("homo")
print({et[1]: int(e.shape[1]) for et, e in G_het["ei"].items()})
print("Số nút:", G_het["n"])


## 7. Mô hình
Trọng số động: `w_r(u,j) = σ(g_r(p_u, φ_r(u,j)))` cho 4 quan hệ (địa bàn, kinh nghiệm, lương, học vấn); điểm = ⟨h_u,h_j⟩ + Σ_r w_r · s_r(u,j).

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import HeteroConv, SAGEConv, GATv2Conv, HGTConv, GCNConv
RELS = ["loc", "exp", "sal", "deg"]

# ----- ngữ cảnh người dùng p_u và đặc trưng tương thích φ_r(u,j) -----
def _nz(x): return torch.nan_to_num(x, nan=0.0)
def _stk(ts): return torch.stack(torch.broadcast_tensors(*ts), -1)
def _unk(*xs):
    m = torch.zeros_like(xs[0], dtype=torch.bool)
    for x in xs: m = m | torch.isnan(x)
    return m.float()
nuni = T["up"].sum(1)
PU = torch.stack([_nz(T["ux"]) / 10, _unk(T["ux"]), _nz(torch.log1p(T["us"])) / 5, _unk(T["us"]), _nz(T["ud"]) / 4, _unk(T["ud"]),
                  nuni.clamp(max=5) / 3, T["uany"], (_nz(T["uage"]).clamp(18, 60) - 18) / 42, T["umar"]], 1)   # [NU, 10]
P_DIM = PU.shape[1]; PHI_DIM = dict(loc=4, exp=2, sal=3, deg=2)

def phis(u, j=None):
    """u: Long[B] ; j=None -> so với toàn bộ job (ma trận B x NJ) ; j: Long[B] -> theo cặp (B,)."""
    mat = j is None
    gu = (lambda k: T[k][u][:, None]) if mat else (lambda k: T[k][u])
    gj = (lambda k: T[k][None]) if mat else (lambda k: T[k][j])
    ov = ((T["up"][u] @ T["jp"].T) > 0).float() if mat else ((T["up"][u] * T["jp"][j]).sum(-1) > 0).float()
    dist = T["dist"][T["uprim"][u][:, None], T["jprim"][None]] if mat else T["dist"][T["uprim"][u], T["jprim"][j]]
    any_ = torch.maximum(gu("uany"), gj("jany")); unk_loc = ((gu("uhas") == 0) | (gj("jhas") == 0)).float()
    ux, jx, us, jh, ud, jd = gu("ux"), gj("jx"), gu("us"), gj("jh"), gu("ud"), gj("jd")
    return dict(
        loc=_stk([ov, any_, dist / 1000, unk_loc]),
        exp=_stk([_nz(((jx - ux) / 5).clamp(-2, 2)), _unk(ux, jx)]),
        sal=_stk([_nz(torch.log((jh + 1) / (us + 1)).clamp(-2, 2)), _unk(us, jh), gj("jneg")]),
        deg=_stk([_nz(((jd - ud) / 4).clamp(-1, 1)), _unk(ud, jd)]))

class Gate(nn.Module):
    """w_r(u,j) = sigmoid(g_r(p_u, φ_r(u,j)))"""
    def __init__(s, pd_, fd, h=16):
        super().__init__(); s.wp = nn.Linear(pd_, h); s.wf = nn.Linear(fd, h); s.out = nn.Linear(h, 1)
    def forward(s, p, phi):                      # p: [...,P], phi: [...,F]; p đã được unsqueeze để broadcast
        return torch.sigmoid(s.out(F.relu(s.wp(p) + s.wf(phi)))).squeeze(-1)

class Net(nn.Module):
    """conv: sage | gat | hgt | homo_sage | homo_gat ; dec: dot | rel_static | rel_dyn"""
    def __init__(s, G, conv="sage", dec="dot", d=64, layers=2, heads=2, drop=0.1):
        super().__init__()
        s.conv_kind, s.dec, s.d, s.drop, s.L, s.kind = conv, dec, d, drop, layers, G["kind"]
        s.lin_u, s.lin_j = nn.Linear(EU.shape[1], d), nn.Linear(EJ.shape[1], d)
        s.attr_emb = nn.ModuleDict({a: nn.Embedding(N_ATTR[a], d) for a in ATTRS}) if G["kind"] == "hetero" else None
        if G["kind"] == "homo":
            s.type_emb = nn.Embedding(len(G["types"]), d)
        ets = list(G["ei"])
        s.convs = nn.ModuleList()
        for _ in range(layers):
            if conv == "sage":   s.convs.append(HeteroConv({et: SAGEConv((d, d), d) for et in ets}, aggr="sum"))
            elif conv == "gat":  s.convs.append(HeteroConv({et: GATv2Conv((d, d), d // heads, heads=heads, edge_dim=1, add_self_loops=False) for et in ets}, aggr="sum"))
            elif conv == "hgt":  s.convs.append(HGTConv(d, d, (G["types"], ets), heads=heads))
            elif conv == "homo_sage": s.convs.append(SAGEConv(d, d))
            elif conv == "homo_gat":  s.convs.append(GATv2Conv(d, d // heads, heads=heads, edge_dim=1, add_self_loops=True))
        if dec != "dot":
            dr = 16
            s.pu = nn.ModuleDict({r: nn.Linear(d, dr) for r in RELS}); s.pj = nn.ModuleDict({r: nn.Linear(d, dr) for r in RELS})
            if dec == "rel_dyn": s.gate = nn.ModuleDict({r: Gate(P_DIM, PHI_DIM[r]) for r in RELS})
            else: s.theta = nn.Parameter(torch.zeros(len(RELS)))
        s.tu = torch.tensor(EU, device=DEV); s.tj = torch.tensor(EJ, device=DEV)

    def embed(s, G):
        hu, hj = s.lin_u(s.tu), s.lin_j(s.tj)
        if G["kind"] == "homo":
            x = torch.cat([hu, hj] + [torch.zeros(G["n"][t], s.d, device=DEV) for t in G["types"][2:]], 0)
            tid = torch.cat([torch.full((G["n"][t],), i, device=DEV) for i, t in enumerate(G["types"])])
            x = x + s.type_emb(tid)
            for c in s.convs:
                out = c(x, G["h_ei"], edge_attr=G["h_ea"]) if s.conv_kind == "homo_gat" else c(x, G["h_ei"])
                x = x + F.dropout(F.relu(out), s.drop, s.training)
            return x[:NU], x[NU:NU + NJ]
        h = {"user": hu, "job": hj}
        if G["kind"] == "hetero":
            for a in ATTRS: h[a] = s.attr_emb[a].weight
        for c in s.convs:
            if s.conv_kind == "gat":   out = c(h, G["ei"], edge_attr_dict=G["ea"])
            else:                      out = c(h, G["ei"])
            h = {t: (h[t] + F.dropout(F.relu(out[t]), s.drop, s.training)) if t in out and out[t] is not None else h[t] for t in h}
        return h["user"], h["job"]

    def score_pairs(s, hu, hj, u, j):
        base = (hu[u] * hj[j]).sum(-1)
        if s.dec == "dot": return base
        ph = phis(u, j) if s.dec == "rel_dyn" else None
        tot = base
        for k, r in enumerate(RELS):
            sr = (s.pu[r](hu[u]) * s.pj[r](hj[j])).sum(-1)
            w = s.gate[r](PU[u], ph[r]) if s.dec == "rel_dyn" else torch.sigmoid(s.theta[k])
            tot = tot + w * sr
        return tot

    def score_matrix(s, hu, hj, u):
        base = hu[u] @ hj.T
        if s.dec == "dot": return base
        ph = phis(u) if s.dec == "rel_dyn" else None
        tot = base
        for k, r in enumerate(RELS):
            sr = s.pu[r](hu[u]) @ s.pj[r](hj).T
            w = s.gate[r](PU[u][:, None, :], ph[r]) if s.dec == "rel_dyn" else torch.sigmoid(s.theta[k])
            tot = tot + w * sr
        return tot

    def gates_pairs(s, u, j):
        ph = phis(u, j); return {r: s.gate[r](PU[u], ph[r]) for r in RELS}

MODELS = {   # tên: (đồ thị, conv, decoder, mô tả)
    "B0_SAGE_2nut":      ("b0",     "sage", "dot",        "đồ thị 2 loại nút, SAGE"),
    "B1_HeteroSAGE":     ("hetero", "sage", "dot",        "6 loại nút, SAGE"),
    "B2_HeteroGATv2":    ("hetero", "gat",  "dot",        "6 loại nút, GATv2 + trọng số cạnh"),
    "B2_HGT":            ("hetero", "hgt",  "dot",        "6 loại nút, HGT"),
    "B3s_TrongSoTinh":   ("hetero", "gat",  "rel_static", "GATv2 + trọng số quan hệ tĩnh (đối chứng)"),
    "B3_TrongSoDong":    ("hetero", "gat",  "rel_dyn",    "GATv2 + trọng số quan hệ động σ(g_r(p_u, φ_r(u,j)))"),
    "H_homo_SAGE":       ("homo",   "homo_sage", "dot",   "đồ thị đồng nhất, SAGE"),
    "H_homo_GAT":        ("homo",   "homo_gat",  "dot",   "đồ thị đồng nhất, GATv2"),
}
GRAPHS = {"b0": G_b0, "hetero": G_het, "homo": G_homo}
def build(name, seed):
    gk, cv, dc, _ = MODELS[name]
    torch.manual_seed(seed); np.random.seed(seed)
    return Net(GRAPHS[gk], cv, dc, CFG["d"], CFG["layers"], CFG["heads"], CFG["dropout"]).to(DEV), GRAPHS[gk]
for n in MODELS:
    m, g = build(n, 0); print(f"{n:20s} params={sum(p.numel() for p in m.parameters())/1e3:.0f}K")


## 8. Huấn luyện và đánh giá
Xếp hạng toàn bộ kho job, loại các cặp dương đã dùng huấn luyện; Recall@20, NDCG@20, MRR, tỷ lệ vi phạm@5/10/20.

In [ ]:
import time
KS = (5, 10, 20)
VCONS = ["province", "experience", "degree", "salary", "any_hard"]
hard_arr = {u: hard[u] for u in keep}
def _excl_lists(sp):
    return (lambda u: np.concatenate([split["train"][u]])) if sp == "val" else (lambda u: np.concatenate([split["train"][u], split["val"][u]]))

@torch.no_grad()
def evaluate(model, G, sp="test", mask=False, pool=0, users_subset=None, bs=32):
    """Xếp hạng toàn bộ job (loại các cặp dương train [+ val]). Trả về dict chỉ số (+ pool ứng viên nếu pool>0)."""
    model.eval(); hu, hj = model.embed(G)
    ex = _excl_lists(sp); users_eval = KEEP if users_subset is None else np.array(users_subset)
    R = {f"viol_{c}@{k}": [] for c in VCONS for k in KS}; R.update(recall20=[], ndcg20=[], mrr=[])
    RB = {c: [] for c in VCONS}; P = dict(u=[], idx=[], sc=[]) if pool else None
    disc = 1.0 / torch.log2(torch.arange(2, 22, device=DEV).float())
    for s0 in range(0, len(users_eval), bs):
        ub = users_eval[s0:s0 + bs]; ut = torch.tensor(ub, device=DEV)
        sc = model.score_matrix(hu, hj, ut).float()
        v = viol(ut); v["any_hard"] = any_hard(v)
        for r, u in enumerate(ub): sc[r, torch.as_tensor(ex(u), device=DEV)] = -1e9
        if mask: sc = sc.masked_fill(v["any_hard"], -1e8)
        top = sc.topk(max(20, pool), dim=1); ti = top.indices
        for c in VCONS: RB[c].append(v[c].float().mean(1).cpu())
        for r, u in enumerate(ub):
            gt = torch.as_tensor(split[sp][u], device=DEV)
            best = sc[r, gt].max(); rank = 1 + (sc[r] > best).sum().item(); R["mrr"].append(1.0 / rank)
            hit = torch.isin(ti[r, :20], gt).float()
            R["recall20"].append(hit.sum().item() / len(gt))
            R["ndcg20"].append(((hit * disc).sum() / disc[:min(len(gt), 20)].sum()).item())
            for c in VCONS:
                vv = v[c][r, ti[r, :20]].float()
                for k in KS: R[f"viol_{c}@{k}"].append(vv[:k].mean().item())
        if pool: P["u"].append(ut); P["idx"].append(ti[:, :pool]); P["sc"].append(top.values[:, :pool])
    out = {k: float(np.mean(x)) for k, x in R.items()}
    out.update({f"random_{c}": float(torch.cat(RB[c]).mean()) for c in VCONS})
    if pool: out["pool"] = {k: torch.cat(x) for k, x in P.items()}
    return out

VAL_USERS = KEEP if len(KEEP) <= 2500 else np.random.RandomState(0).choice(KEEP, 2500, replace=False)

def sample_batch(pairs, nj_hard=True):
    u = torch.tensor(pairs[:, 0], device=DEV); jp = torch.tensor(pairs[:, 1], device=DEV)
    jr = torch.randint(0, NJ, (len(pairs),), device=DEV)
    jh_ = np.array([np.random.choice(hard_arr[a]) if len(hard_arr[a]) else np.random.randint(NJ) for a in pairs[:, 0]])
    return u, jp, jr, torch.tensor(jh_, device=DEV)

def train_model(name, seed, verbose=True):
    model, G = build(name, seed)
    opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"], eta_min=CFG["lr"] * 0.05)   # giảm dần lr cho ổn định
    best, best_state, bad, ep_times = -1, None, 0, []
    for ep in range(1, CFG["epochs"] + 1):
        model.train(); t0 = time.time(); perm = np.random.permutation(len(tr_pairs)); tot, nst = 0, 0
        for s0 in range(0, len(perm), CFG["batch"]):
            u, jp, jr, jhd = sample_batch(tr_pairs[perm[s0:s0 + CFG["batch"]]])
            hu, hj = model.embed(G)
            sp_, sr_, sh_ = model.score_pairs(hu, hj, u, jp), model.score_pairs(hu, hj, u, jr), model.score_pairs(hu, hj, u, jhd)
            loss = F.binary_cross_entropy_with_logits(torch.cat([sp_, sr_, sh_]), torch.cat([torch.ones_like(sp_), torch.zeros_like(sr_), torch.zeros_like(sh_)]))
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item(); nst += 1
        sched.step()
        if DEV.type == "cuda": torch.cuda.synchronize()
        ep_times.append(time.time() - t0)
        if ep % CFG["eval_every"] == 0 or ep == CFG["epochs"]:
            m = evaluate(model, G, "val", users_subset=VAL_USERS)
            if verbose: print(f"  [{name} s{seed}] ep{ep:3d} loss={tot / nst:.4f} val NDCG@20={m['ndcg20']:.4f} R@20={m['recall20']:.4f} MRR={m['mrr']:.4f}")
            if m["ndcg20"] > best: best, bad, best_state = m["ndcg20"], 0, copy.deepcopy(model.state_dict())
            else:
                bad += 1
                if bad >= CFG["patience"]: break
    model.load_state_dict(best_state)
    return model, G, dict(epoch_time=float(np.mean(ep_times)), epochs=len(ep_times), best_val_ndcg=best)


## 9. Chạy B0 → B3 (nhiều seed) và đo độ phức tạp
Kèm 2 mốc tham chiếu không học sâu, cùng giao thức đánh giá: `REF_Popularity` (số lượt ứng tuyển train) và `REF_UserKNN` (lọc cộng tác theo ứng viên tương tự).

In [ ]:
rows, trained, cost = [], {}, []

class FixedScorer:
    """Bọc một hàm điểm cố định để dùng chung evaluate()."""
    def __init__(s, fn): s.fn = fn
    def eval(s): return s
    def embed(s, G): return None, None
    def score_matrix(s, hu, hj, u): return s.fn(u)
_deg = torch.zeros(NJ, device=DEV).index_add_(0, torch.tensor(tr_pairs[:, 1], device=DEV), torch.ones(len(tr_pairs), device=DEV))
_R = torch.sparse_coo_tensor(torch.tensor(tr_pairs.T, device=DEV), torch.ones(len(tr_pairs), device=DEV), (NU, NJ)).coalesce()
_UU = torch.sparse.mm(_R, _R.t().to_dense())                     # NU x NU: số job ứng tuyển chung
_UU.fill_diagonal_(0); _RT = _R.t().coalesce()
REFS = {"REF_Popularity": lambda u: _deg[None].expand(len(u), -1).clone(),
        "REF_UserKNN": lambda u: torch.sparse.mm(_RT, _UU[u].T).T + 1e-3 * _deg[None]}
for name, fn in REFS.items():
    for variant, mk in (("raw", False), ("+mask", True)):
        m = evaluate(FixedScorer(fn), None, "test", mask=mk); m.pop("pool", None)
        rows.append(dict(model=name, seed=0, variant=variant, **m))
    print(f"{name:16s} test NDCG@20={rows[-2]['ndcg20']:.4f} R@20={rows[-2]['recall20']:.4f}  (+mask NDCG@20={rows[-1]['ndcg20']:.4f})")
for name in MODELS:
    for seed in CFG["seeds"]:
        if DEV.type == "cuda": torch.cuda.reset_peak_memory_stats()
        t0 = time.time(); model, G, info = train_model(name, seed)
        for variant, mk in (("raw", False), ("+mask", True)):
            m = evaluate(model, G, "test", mask=mk); m.pop("pool", None)
            rows.append(dict(model=name, seed=seed, variant=variant, **m))
        # chi phí
        ub = torch.tensor(KEEP[:32], device=DEV)
        if DEV.type == "cuda": torch.cuda.synchronize()
        t1 = time.time()
        with torch.no_grad():
            model.eval(); hu, hj = model.embed(G); _ = model.score_matrix(hu, hj, ub).topk(20, 1)
        if DEV.type == "cuda": torch.cuda.synchronize()
        cost.append(dict(model=name, seed=seed, params=sum(p.numel() for p in model.parameters()), sec_per_epoch=info["epoch_time"],
                         epochs=info["epochs"], infer_ms_per_user=(time.time() - t1) * 1000 / len(ub),
                         gpu_mem_MB=(torch.cuda.max_memory_allocated() / 2**20 if DEV.type == "cuda" else np.nan), total_sec=time.time() - t0))
        if seed == CFG["seeds"][0]: trained[name] = (model, G)
        print(f"== {name} seed={seed} xong ({time.time()-t0:.0f}s)")
RES = pd.DataFrame(rows); COST = pd.DataFrame(cost)
RES.to_csv(OUT / "results_all_runs.csv", index=False); COST.to_csv(OUT / "complexity.csv", index=False)
agg = RES.groupby(["model", "variant"]).agg(["mean", "std"]).drop(columns="seed", level=0)
agg.to_csv(OUT / "results_mean_std.csv")
show = ["recall20", "ndcg20", "mrr", "viol_any_hard@5", "viol_any_hard@10", "viol_any_hard@20", "viol_province@20", "viol_experience@20", "viol_degree@20", "viol_salary@20"]
print(RES.groupby(["model", "variant"])[show].mean().round(4).to_string())
print("\nMốc ngẫu nhiên (tỷ lệ vi phạm khi xếp ngẫu nhiên, trên dữ liệu mới):")
print(RES[[c for c in RES.columns if c.startswith("random_")]].mean().round(4).to_string())


## 10. B4: lọc cứng + xếp hạng lại (λ chọn trên validation)

In [ ]:
# ===== B4: B3 + lọc cứng + xếp hạng lại đa tiêu chí =====
#   (a) tuyến tính: điểm GNN chuẩn hóa − phạt lương/dư KN/thừa bằng cấp + thưởng đãi ngộ, λ chọn trên validation
#   (b) LambdaRank (LightGBM) học trên các ứng viên top-200 của B3
import lightgbm as lgb
m3, G3 = trained["B3_TrongSoDong"]
POOL = 200; disc20 = 1.0 / torch.log2(torch.arange(2, 22, device=DEV).float()); ICG = disc20.cumsum(0)
pool_val = evaluate(m3, G3, "val", mask=True, pool=POOL); pool_test = evaluate(m3, G3, "test", mask=True, pool=POOL)
ex_fn = {"val": _excl_lists("val"), "test": _excl_lists("test")}
FEAT_NAMES = ["z", "raw", "rank", "sal_short", "overqual", "deg_surplus", "benefit", "benefit_known", "exp_gap", "deg_gap", "sal_logratio", "dist_km", "same_unit", "job_any", "sal_negotiable", "u_exp", "u_deg"]

def feats(u, idx, sc):
    """u:[n] ; idx,sc:[n,P] (sc = điểm B3, -1e8 nếu bị lọc). Trả về dict đặc trưng cho từng (ứng viên, job ứng viên)."""
    ok = sc > -1e7; n_ok = ok.sum(1, keepdim=True).clamp(min=1)
    mu = (sc * ok).sum(1, keepdim=True) / n_ok; sd = (((sc - mu).pow(2) * ok).sum(1, keepdim=True) / n_ok).sqrt().clamp(min=1e-6); z = (sc - mu) / sd
    us_, ux_, ud_ = T["us"][u][:, None], T["ux"][u][:, None], T["ud"][u][:, None]
    jh_, jx_, jd_ = T["jh"][idx], T["jx"][idx], T["jd"][idx]
    sf = _nz(F.relu((us_ - jh_) / us_.clamp(min=1e-3))); oq = _nz(F.relu(ux_ - jx_ - 2) / 5); ds = _nz(F.relu(ud_ - jd_) / 4)
    ben, benk = T["jb"][idx], T["jbk"][idx]
    ov = ((T["up"][u][:, None, :] * T["jp"][idx]).sum(-1) > 0).float()
    dist = T["dist"][T["uprim"][u][:, None], T["jprim"][idx]] / 1000
    cols = [z, sc.clamp(min=-50), torch.arange(sc.shape[1], device=DEV).float().expand_as(sc) / sc.shape[1], sf, oq, ds, ben, benk,
            _nz(((jx_ - ux_) / 5).clamp(-2, 2)), _nz(((jd_ - ud_) / 4).clamp(-1, 1)), _nz(torch.log((jh_ + 1) / (us_ + 1)).clamp(-2, 2)), dist, ov,
            T["jany"][idx], T["jneg"][idx], ux_.nan_to_num(0).expand_as(sc) / 10, ud_.nan_to_num(0).expand_as(sc) / 4]
    return dict(ok=ok, z=z, sf=sf, oq=oq, ds=ds, ben=ben, X=torch.stack(cols, -1))

def lin_scores(Fd, lam): return (Fd["z"] - lam[0] * Fd["sf"] - lam[1] * Fd["oq"] - lam[2] * Fd["ds"] + lam[3] * Fd["ben"]).masked_fill(~Fd["ok"], -1e9)

def prep(res, sp):
    P = res["pool"]; u, idx = P["u"], P["idx"]; Fd = feats(u, idx, P["sc"])
    H = torch.zeros_like(idx, dtype=torch.bool); ng = torch.zeros(len(u), device=DEV)
    for r in range(len(u)):
        gt = torch.as_tensor(split[sp][int(u[r])], device=DEV); H[r] = torch.isin(idx[r], gt); ng[r] = len(gt)
    v = viol(u[:1]); V = {c: [] for c in VCONS}
    for s0 in range(0, len(u), 64):
        vv = viol(u[s0:s0 + 64]); vv["any_hard"] = any_hard(vv)
        for c in VCONS: V[c].append(vv[c].gather(1, idx[s0:s0 + 64]))
    return dict(Fd=Fd, H=H, ng=ng, V={c: torch.cat(x) for c, x in V.items()}, u=u, idx=idx)

def eval_scores(D, s):
    order = s.argsort(1, descending=True); h = D["H"].gather(1, order).float(); ng = D["ng"]
    out = dict(recall20=float((h[:, :20].sum(1) / ng).mean()), ndcg20=float(((h[:, :20] * disc20).sum(1) / ICG[(ng.clamp(max=20) - 1).long()]).mean()))
    first = (h.cumsum(1) > 0).float().argmax(1); out["mrr"] = float(torch.where(h.sum(1) > 0, 1.0 / (first + 1.0), torch.zeros_like(first, dtype=torch.float)).mean())  # dương ngoài top-200 coi MRR≈0
    for c in VCONS:
        vv = D["V"][c].gather(1, order).float()
        for k in KS: out[f"viol_{c}@{k}"] = float(vv[:, :k].mean())
    return out
Dv, Dt = prep(pool_val, "val"), prep(pool_test, "test")

# (a) tuyến tính
grid = [(a, b, c, e) for a in (0, .5, 1, 2) for b in (0, .1, .3) for c in (0, .1, .3) for e in (0, .1, .3)]
best_lin = max((eval_scores(Dv, lin_scores(Dv["Fd"], lam))["ndcg20"], lam) for lam in grid); LAM = best_lin[1]
print("Tuyến tính: λ (lương, dư KN, dư bằng, đãi ngộ) =", LAM, "| val NDCG@20 =", round(best_lin[0], 4))

# (b) LambdaRank: huấn luyện trên ứng viên top-200 của tập validation (80%), dừng sớm trên 20% còn lại
def rows(D, sel):
    X, y, g = [], [], []
    for r in sel:
        ok = D["Fd"]["ok"][r]; X.append(D["Fd"]["X"][r][ok].cpu().numpy()); y.append(D["H"][r][ok].cpu().numpy().astype(int)); g.append(int(ok.sum()))
    return np.concatenate(X), np.concatenate(y), g
rp = np.random.RandomState(0).permutation(len(Dv["u"])); ntr = int(.8 * len(rp))
Xa, ya, ga = rows(Dv, rp[:ntr]); Xb, yb, gb = rows(Dv, rp[ntr:])
RANKER = lgb.LGBMRanker(objective="lambdarank", n_estimators=400, learning_rate=0.05, num_leaves=15, min_child_samples=20, subsample=0.8, subsample_freq=1,
                        colsample_bytree=0.8, lambdarank_truncation_level=20, verbose=-1, random_state=0)
RANKER.fit(Xa, ya, group=ga, eval_set=[(Xb, yb)], eval_group=[gb], eval_at=[20], callbacks=[lgb.early_stopping(30, verbose=False)])
print("LambdaRank: số cây =", RANKER.best_iteration_ or RANKER.n_estimators, "| mẫu huấn luyện:", len(ya), "| dương:", int(ya.sum()))
imp = pd.Series(RANKER.feature_importances_, index=FEAT_NAMES).sort_values(ascending=False); print("Độ quan trọng đặc trưng (top 6):", imp.head(6).to_dict())

def rank_scores(Fd):
    s = torch.full(Fd["ok"].shape, -1e9, device=DEV); ok = Fd["ok"]
    if ok.any(): s[ok] = torch.tensor(RANKER.predict(Fd["X"][ok].cpu().numpy()), device=DEV, dtype=torch.float)
    return s
def rerank_scores(kind, Fd):
    return Fd["z"].masked_fill(~Fd["ok"], -1e9) if kind == "none" else (lin_scores(Fd, LAM) if kind == "linear" else rank_scores(Fd))
VAL_SC = {k: eval_scores(Dv, rerank_scores(k, Dv["Fd"]))["ndcg20"] for k in ("linear", "lambdarank")}
BEST = max(VAL_SC, key=VAL_SC.get); print("Val NDCG@20:", {k: round(v, 4) for k, v in VAL_SC.items()}, "-> B4 dùng:", BEST)

names = {"none": "B3 + lọc (không xếp hạng lại)", "linear": "B4a = B3 + lọc + xếp hạng lại tuyến tính", "lambdarank": "B4b = B3 + lọc + LambdaRank"}
B4 = pd.DataFrame([dict(model=names[k], **eval_scores(Dt, rerank_scores(k, Dt["Fd"]))) for k in ("none", "linear", "lambdarank")])
B4.to_csv(OUT / "b4_rerank.csv", index=False); imp.to_csv(OUT / "lambdarank_feature_importance.csv")
print(B4[["model", "recall20", "ndcg20", "mrr", "viol_any_hard@20", "viol_salary@5", "viol_salary@10", "viol_salary@20"]].round(4).to_string(index=False))


## 11. Cắt tỉa cạnh · trọng số động học được · dị thể vs đồng nhất

In [ ]:
# ===== (1) Cắt tỉa cạnh theo phân vị trọng số =====
def edge_weights(model):
    with torch.no_grad():
        model.eval(); g = {r: [] for r in RELS}
        for s0 in range(0, len(tr_pairs), 8192):
            b = torch.tensor(tr_pairs[s0:s0 + 8192], device=DEV); gg = model.gates_pairs(b[:, 0], b[:, 1])
            for r in RELS: g[r].append(gg[r])
        g = {r: torch.cat(v) for r, v in g.items()}
        uidx, jidx = torch.tensor(tr_pairs[:, 0], device=DEV), torch.tensor(tr_pairs[:, 1], device=DEV)
        W = {}
        for r in RELS:
            for side, idx, n in (("user", uidx, NU), ("job", jidx, NJ)):
                sm = torch.zeros(n, device=DEV).index_add_(0, idx, g[r]); ct = torch.zeros(n, device=DEV).index_add_(0, idx, torch.ones_like(g[r]))
                W[(side, r)] = torch.where(ct > 0, sm / ct.clamp(min=1), torch.ones_like(sm)).cpu().numpy()
    return W
EW = edge_weights(m3)
def _keep(w, q, r_):
    n = len(w); keep = np.ones(n, bool)
    if q > 0 and n:
        order = np.argsort(w + 1e-9 * r_.rand(n), kind="stable"); keep[order[:int(q * n)]] = False   # bỏ đúng q% cạnh trọng số thấp nhất
    return keep
def keep_masks(q, mode, seed=0):
    r_ = np.random.RandomState(seed); km = {}
    for (side, a), (s, d) in CAN.items():
        km[(side, a)] = _keep((EW[(side, a)][s] if (side, a) in EW else np.ones(len(s))) if mode == "weight" else r_.rand(len(s)), q, r_)
    for a, (s, d, w) in NEAR.items():
        km[("near", a)] = _keep(w if mode == "weight" else r_.rand(len(s)), q, r_)
    return km
prune_rows = []
for mode in ("weight", "random"):
    for q in (0.0, 0.1, 0.3, 0.5, 0.7):
        if q == 0 and mode == "random": continue
        km = keep_masks(q, mode); Gp = make_graph("hetero", km)
        ne = sum(int(v.sum()) for v in km.values()); ne0 = sum(len(v) for v in km.values())
        t0 = time.time(); m = evaluate(m3, Gp, "test"); m.pop("pool", None); dt = time.time() - t0
        prune_rows.append(dict(mode=mode, prune_q=q, edges_kept=f"{ne}/{ne0}", eval_sec=dt, recall20=m["recall20"], ndcg20=m["ndcg20"], mrr=m["mrr"],
                               viol_any_hard20=m["viol_any_hard@20"]))
PRUNE = pd.DataFrame(prune_rows); PRUNE.to_csv(OUT / "pruning.csv", index=False)
print("Cắt tỉa cạnh (B3, không huấn luyện lại; 'weight' = bỏ cạnh có trọng số thấp nhất, 'random' = đối chứng):"); print(PRUNE.round(4).to_string(index=False))

# ===== (2) Trọng số động học được: trung bình cổng theo quan hệ, cặp khả thi vs cặp vi phạm =====
with torch.no_grad():
    m3.eval(); rs = np.random.RandomState(1); ub = rs.choice(KEEP, min(len(KEEP), 300), replace=False)
    pairs = [(u, j, 1) for u in ub for j in split["test"][u][:3]] + [(u, j, 0) for u in ub for j in hard_arr[u][:3]]
    pu, pj, pl = (torch.tensor(np.array(x), device=DEV) for x in zip(*pairs)); gg = m3.gates_pairs(pu, pj)
    GATES = pd.DataFrame({r: [float(gg[r][pl == 1].mean()), float(gg[r][pl == 0].mean())] for r in RELS}, index=["cặp khả thi (dương)", "cặp vi phạm (negative khó)"])
GATES.to_csv(OUT / "gate_means.csv"); print("\nTrọng số động trung bình theo quan hệ:"); print(GATES.round(3).to_string())

# ===== (3) Đồ thị dị thể vs đồng nhất =====
base = RES[RES.variant == "raw"].groupby("model")[["recall20", "ndcg20", "mrr", "viol_any_hard@20"]].mean()
cc = COST.groupby("model")[["params", "sec_per_epoch", "infer_ms_per_user"]].mean()
HH = base.join(cc).loc[["B1_HeteroSAGE", "H_homo_SAGE", "B2_HeteroGATv2", "H_homo_GAT"]]
HH.to_csv(OUT / "hetero_vs_homo.csv"); print("\nDị thể vs đồng nhất (cùng cấu trúc cạnh, cùng bộ nhãn):"); print(HH.round(4).to_string())


## 12. Bộ nhãn tay
Xuất `handlabel_to_label.csv` (đã ẩn tên mô hình, trộn thứ tự). Điền cột `label` (0/1/2), tải lên làm Kaggle dataset rồi đặt `HANDLABEL_CSV` và chạy lại cell này.

In [ ]:
# ===== Bộ nhãn tay (đánh giá độc lập với nhãn sinh tự động) =====
rs = np.random.RandomState(7); HU = rs.choice(KEEP, min(len(KEEP), CFG["n_handlabel_users"]), replace=False)
def top_lists(model, G, k=10):
    out = {}
    with torch.no_grad():
        model.eval(); hu, hj = model.embed(G); ex = ex_fn["test"]
        for s0 in range(0, len(HU), 32):
            ub = HU[s0:s0 + 32]; ut = torch.tensor(ub, device=DEV); sc = model.score_matrix(hu, hj, ut).float()
            for r, u in enumerate(ub): sc[r, torch.as_tensor(ex(u), device=DEV)] = -1e9
            for r, u in enumerate(ub): out[int(u)] = sc[r].topk(k).indices.tolist()
    return out
TOPS = {n: top_lists(*trained[n]) for n in ("B1_HeteroSAGE", "B2_HeteroGATv2", "B3_TrongSoDong")}
# B4a/B4b: lọc + xếp hạng lại (theo pool test đã tính ở trên)
u_pool = {int(a): i for i, a in enumerate(Dt["u"].cpu().numpy())}
for kind, nm in (("linear", "B4a_XepHangLaiTuyenTinh"), ("lambdarank", "B4b_LambdaRank")):
    sc_ = rerank_scores(kind, Dt["Fd"]); top = Dt["idx"].gather(1, sc_.argsort(1, descending=True))[:, :10].cpu().numpy()
    TOPS[nm] = {int(u): top[u_pool[int(u)]].tolist() for u in HU}
json.dump({m: {str(u): v for u, v in d.items()} for m, d in TOPS.items()}, open(OUT / "handlabel_tops.json", "w"))
rows_ = []
for u in HU:
    js = sorted({j for m in TOPS for j in TOPS[m][int(u)]}); rs.shuffle(js)
    ur = raw_users.loc[int(u)]
    desc = f"{ur['desired_job']} | {ur['industry']} | nơi muốn làm: {ur['workplace_raw']} | KN: {ur['experience_raw']} | lương: {ur['salary_raw']} | bằng: {str(ur['degree_raw'])[:80]}"
    for j in js:
        rj = raw_jobs.loc[j]
        sal = "thỏa thuận" if rj["sal_kind"] in ("negotiable", "missing") else f"{rj['sal_min']}-{rj['sal_max']} triệu"
        rows_.append(dict(user_id=ur["user_id"], u_idx=int(u), user_desc=desc, j_idx=int(j), job_title=rj["title"], job_location=rj["units"] or rj["address"],
                          job_exp=rj["exp_min"], job_salary=sal, job_quals=DEGREE_NAMES.get(int(rj["deg_min"])) if pd.notna(rj["deg_min"]) else "không nêu", label=""))
HL = pd.DataFrame(rows_); HL.to_csv(OUT / "handlabel_to_label.csv", index=False, encoding="utf-8-sig")
print(f"Đã xuất {len(HL)} cặp cần gán nhãn cho {len(HU)} ứng viên -> handlabel_to_label.csv (điền cột label: 0 = không phù hợp, 1 = phù hợp một phần, 2 = phù hợp)")

def score_handlabels(path):
    L = pd.read_csv(path); L = L[pd.to_numeric(L["label"], errors="coerce").notna()].copy(); L["label"] = L["label"].astype(float)
    lab = {(int(a), int(b)): c for a, b, c in zip(L["u_idx"], L["j_idx"], L["label"])}; out = {}
    for m, d in TOPS.items():
        ps, pl, nd = [], [], []
        for u, js in d.items():
            ls = [lab.get((u, j)) for j in js]
            if any(x is None for x in ls): continue
            ps.append(np.mean([x == 2 for x in ls])); pl.append(np.mean([x >= 1 for x in ls]))
            g = np.array(ls); dcg = (g / np.log2(np.arange(2, len(g) + 2))).sum(); ig = (np.sort(g)[::-1] / np.log2(np.arange(2, len(g) + 2))).sum(); nd.append(dcg / ig if ig > 0 else 0)
        out[m] = dict(users=len(ps), P10_strict=np.mean(ps) if ps else np.nan, P10_lenient=np.mean(pl) if pl else np.nan, NDCG10_graded=np.mean(nd) if nd else np.nan)
    return pd.DataFrame(out).T
HL_IN = os.environ.get("HANDLABEL_CSV")
if HL_IN and os.path.exists(HL_IN):
    HLR = score_handlabels(HL_IN); HLR.to_csv(OUT / "handlabel_scores.csv"); print(HLR.round(3).to_string())
else:
    print("Chưa có file nhãn tay. Sau khi điền nhãn: đặt HANDLABEL_CSV=<đường dẫn> rồi chạy lại cell này (hoặc gọi score_handlabels(path)).")


## 13. Tổng hợp & xuất kết quả

In [ ]:
# ===== Tổng hợp + xuất =====
final = RES[RES.variant == "raw"].groupby("model")[["recall20", "ndcg20", "mrr", "viol_any_hard@5", "viol_any_hard@10", "viol_any_hard@20"]].agg(["mean", "std"])
final.to_csv(OUT / "table_main_raw.csv")
summary = dict(dataset=str(DATA_DIR), dataset_baseline_check=META.get("baseline_check"), n_jobs=NJ, n_users_total=NU, n_users_labeled=len(KEEP), n_train_pairs=int(len(tr_pairs)), cfg=CFG, lam_rerank=list(LAM),
               random_baseline={c: float(RES[f"random_{c}"].mean()) for c in VCONS}, device=str(DEV), smoke=SMOKE)
json.dump(summary, open(OUT / "summary.json", "w"), ensure_ascii=False, indent=1, default=str)
print(json.dumps(summary["random_baseline"], ensure_ascii=False)); print("Các file kết quả:", sorted(p.name for p in OUT.iterdir()))
if IN_KAGGLE:
    import shutil; shutil.make_archive(str(WORK / "results_bundle"), "zip", OUT); print("Đã nén ->", WORK / "results_bundle.zip")


## 14. Xuất artifact cho web
Mô hình, embedding, chỉ số ánh xạ (kèm `db_job_id` / `db_user_id` của backend), bảng job/user và top-100 đã lọc + xếp hạng lại (B4) cho mọi ứng viên -> `results/artifacts/`.

In [ ]:
# ===== Xuất artifact cho web (mô hình, embedding, bảng ánh xạ, top-K sẵn) =====
ART = OUT / "artifacts"; ART.mkdir(exist_ok=True)
SERVE_NAME = "B3_TrongSoDong"            # mô hình dùng cho B4 (đổi nếu kết quả cho thấy mô hình khác tốt hơn)
ms, Gs = trained[SERVE_NAME]

@torch.no_grad()
def serve_topk(model, G, uidx, kind, k=100, pool=300, bs=32):
    """B4 cho mọi ứng viên (không loại cặp nào): điểm B3 -> lọc cứng -> top-pool -> xếp hạng lại (kind) -> top-k."""
    model.eval(); hu, hj = model.embed(G); out_i, out_s = [], []
    for s0 in range(0, len(uidx), bs):
        ut = torch.tensor(uidx[s0:s0 + bs], device=DEV)
        sc = model.score_matrix(hu, hj, ut).float(); v = viol(ut); sc = sc.masked_fill(any_hard(v), -1e8)
        top = sc.topk(min(pool, NJ), 1); fin = rerank_scores(kind, feats(ut, top.indices, top.values))
        o = fin.argsort(1, descending=True)[:, :k]; out_i.append(top.indices.gather(1, o).cpu()); out_s.append(fin.gather(1, o).cpu())
    return torch.cat(out_i).numpy(), torch.cat(out_s).numpy()

ALL_U = np.arange(NU); TI, TS = serve_topk(ms, Gs, ALL_U, BEST)
with torch.no_grad():
    ms.eval(); HU_, HJ_ = ms.embed(Gs)
np.save(ART / "emb_user_gnn.npy", HU_.cpu().numpy()); np.save(ART / "emb_job_gnn.npy", HJ_.cpu().numpy())
np.save(ART / "text_emb_user.npy", EU); np.save(ART / "text_emb_job.npy", EJ)
RANKER.booster_.save_model(str(ART / "lambdarank.txt"))
torch.save(dict(name=SERVE_NAME, spec=MODELS[SERVE_NAME][:3], state=ms.state_dict(), cfg=dict(d=CFG["d"], layers=CFG["layers"], heads=CFG["heads"], dropout=CFG["dropout"])), ART / "model.pt")
np.savez(ART / "graph.npz", tr_pairs=tr_pairs, **{f"can_{s}_{a}_src": v[0] for (s, a), v in CAN.items()}, **{f"can_{s}_{a}_dst": v[1] for (s, a), v in CAN.items()},
         **{f"near_{a}_src": v[0] for a, v in NEAR.items()}, **{f"near_{a}_dst": v[1] for a, v in NEAR.items()}, **{f"near_{a}_w": v[2] for a, v in NEAR.items()})
np.savez(ART / "topk_b4.npz", user_idx=ALL_U, job_idx=TI, score=TS)

_cols_j = ["job_id", "db_job_id", "db_job_ids", "title", "company", "units", "occupation", "industry_group", "level", "employment_type",
           "exp_min", "sal_min", "sal_max", "sal_kind", "deg_min"]
jt = raw_jobs[_cols_j].rename(columns={"sal_min": "sal_min_trieu", "sal_max": "sal_max_trieu"}); jt.insert(0, "j_idx", np.arange(NJ))
jt.to_csv(ART / "jobs.csv", index=False, encoding="utf-8-sig")
_cols_u = ["user_id", "db_user_id", "db_user_ids", "desired_job", "industry_group", "occupation", "units", "exp_max", "sal_min", "deg_max", "age"]
ut_ = raw_users[_cols_u].rename(columns={"sal_min": "sal_min_trieu"}); ut_.insert(0, "u_idx", np.arange(NU)); ut_.insert(4, "has_labels", [u in set(keep) for u in range(NU)])
ut_.to_csv(ART / "users.csv", index=False, encoding="utf-8-sig")
json.dump(dict(model=SERVE_NAME, lam_rerank=list(LAM), rerank_kind=BEST, pool=300, topk=100, emb_model=CFG["emb_model"], text_emb_dim=int(EU.shape[1]), n_users=NU, n_jobs=NJ,
               unit_names=UNIT_NAMES, unit_latlon=[UNITS[n][:2] for n in UNIT_NAMES], loc_any=LOC_ANY, n_attr=N_ATTR, rels=RELS, exp_tol=CFG["exp_tol"], sal_ratio=CFG["sal_ratio"],
               buckets=dict(exp="0 | 1-2 | 3-5 | 6-10 | >10 | không rõ", sal_trieu="<5 | 5-10 | 10-15 | 15-20 | 20-30 | 30-50 | >50 | thỏa thuận/không rõ", deg=DEGREE_NAMES),
               feature_names=FEAT_NAMES, data=dict(dataset=META["name"], seed=META["seed"], backend_mapping=META["backend_mapping"]), note="Top-K đã lọc cứng (tỉnh/kinh nghiệm/bằng cấp) + xếp hạng lại; job_idx/u_idx là chỉ số hàng trong jobs.csv/users.csv"),
          open(ART / "meta.json", "w"), ensure_ascii=False, indent=1, default=str)
print("Artifact:", {p.name: f"{p.stat().st_size/1e6:.1f} MB" for p in sorted(ART.iterdir())})
# kiểm tra nhanh: top-K của một ứng viên
u0 = int(KEEP[0]); print("Ví dụ:", ut_.loc[u0, ["desired_job", "units", "exp_max", "sal_min_trieu"]].to_dict())
print(jt.loc[TI[u0][:5], ["title", "units", "exp_min", "sal_min_trieu", "sal_max_trieu"]].to_string())
